# Titanium sample experiment inventory — HTMDEC / AIMD-L

Find titanium-related IGSN titles, include their child IGSNs, and count distinct stored files in raw XRD and PDV items by experiment date.

**Start here:** Save this notebook in `aimdl_examples/tutorials`, select your existing Python environment, and run the cells in order (or Run All). It reads `GIRDER_API_URL` and `GIRDER_API_KEY` from the nearest `.env` in the current directory or its parents. If no key is available, it prompts privately. No prior tutorial cells are needed. Python 3.9+.

**Scope:** title matching identifies candidates, not all titanium-bearing materials. Counts cover accessible AIMD-L `xrd_raw` and `pdv_trace` items, exclude processed products and calibrants, and count stored file IDs rather than shots or experiments. Children inherit a candidate classification from the closest matching ancestor title; this is explicitly recorded. No experimental file bytes are downloaded and no server records are modified.

**Dates:** use the calendar date as recorded in `meta.experiment_date`, with no timezone conversion or substitution of upload dates. Unknown/invalid dates are retained. Only dates with these raw-data types appear.

This notebook was prepared against the repository tutorial and observed API behavior. It avoids the unsupported `$regex` filter. Authenticated execution must take place in your environment; local validation uses simulated API responses.


## 1. Dependencies
Run this optional cell only if imports in the next cell fail; uncomment its line. Then rerun the imports. No packages are installed automatically.


In [ ]:
# %pip install pandas girder-client python-dotenv


In [1]:
import os
import json
import time
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd
from dotenv import load_dotenv
from girder_client import GirderClient, HttpError
from IPython.display import display


## 2. Configuration and authentication
Defaults implement the requested title search. `Ti` matching is case-sensitive to avoid matching lowercase letters inside “synthesis”; “titanium” is case-insensitive. Add known sample IDs or exclude false positives below. Setting `INCLUDE_CHILDREN=False` requires exact IGSN matches.

Outputs go to a new timestamped directory on every run. File enumeration may take several minutes because it requests metadata for every matching item. Progress is printed; the completed file listings remain in memory if interrupted and resumed in the same kernel.


In [2]:
API_URL_OVERRIDE = None   # Optional explicit API URL
ENV_FILE = None           # Optional explicit path to your .env file
PAGE_SIZE = 100
INCLUDE_CHILDREN = True
TITLE_TERMS_CASE_SENSITIVE = ["Ti"]
TITLE_TERMS_CASE_INSENSITIVE = ["titanium"]
EXTRA_IGSNS = []          # Exact IGSNs to include even without a title match
EXCLUDE_IGSNS = []        # Exclude these IGSNs and their descendants
DATA_TYPES = {"xrd_raw": "XRD", "pdv_trace": "PDV"}
OUTPUT_BASE = Path.cwd() / "titanium_inventory_outputs"

if ENV_FILE:
    env_path = Path(ENV_FILE).expanduser()
    if not env_path.is_file():
        raise FileNotFoundError(env_path)
else:
    env_path = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / ".env").is_file()), None)
if env_path:
    load_dotenv(env_path, override=False)

api_url = (API_URL_OVERRIDE or os.getenv("GIRDER_API_URL")
           or "https://data.htmdec.org/api/v1").rstrip("/")
gc = GirderClient(apiUrl=api_url)
api_key = os.getenv("GIRDER_API_KEY") or getpass("Girder API key: ")
try:
    gc.authenticate(apiKey=api_key)
except Exception:
    raise RuntimeError("Authentication failed. Check the API URL and API key.") from None
finally:
    del api_key
user = gc.get("user/me")
print("Authenticated as:", user["login"])
print("API:", api_url)


Authenticated as: elbert
API: https://data.htmdec.org/api/v1


## 3. API pagination and matching helpers
No regex or other server-side metadata filters are used. HTTP failures stop the run rather than producing a silently incomplete report. Temporary HTTP 429/5xx failures receive a few short retries. Pagination continues to an empty page and detects repeated pages.


In [3]:
def api_get(endpoint, parameters=None):
    for attempt in range(4):
        try:
            return gc.get(endpoint, parameters=parameters)
        except HttpError as exc:
            status = getattr(exc, "status", None)
            if status in (429, 500, 502, 503, 504) and attempt < 3:
                time.sleep(2 ** attempt)
                continue
            raise


def api_pages(endpoint, parameters=None):
    offset, seen = 0, set()
    while True:
        batch = api_get(endpoint, {
            **(parameters or {}), "limit": PAGE_SIZE,
            "offset": offset, "sort": "_id", "sortdir": 1,
        })
        if not isinstance(batch, list):
            raise TypeError(f"{endpoint} returned {type(batch).__name__}, expected list")
        if not batch:
            return
        new_count = 0
        for record in batch:
            identifier = record["_id"]
            if identifier not in seen:
                seen.add(identifier)
                new_count += 1
                yield record
        if not new_count:
            raise RuntimeError(f"Repeated page from {endpoint}; pagination stopped")
        offset += len(batch)


def clean_igsn(value):
    return value.strip() if isinstance(value, str) else ""


def is_descendant(igsn, parent):
    return igsn == parent or (INCLUDE_CHILDREN and igsn.startswith(parent + "-"))


def excluded(igsn):
    return any(igsn == x or igsn.startswith(x + "-") for x in EXCLUDE_IGSNS)


def title_matches(title):
    return (any(term in title for term in TITLE_TERMS_CASE_SENSITIVE)
            or any(term.lower() in title.lower()
                   for term in TITLE_TERMS_CASE_INSENSITIVE))


def experiment_day(value):
    if value is None or str(value).strip() == "":
        return "UNKNOWN_DATE"
    try:
        return datetime.fromisoformat(str(value).strip().replace("Z", "+00:00")).date().isoformat()
    except (ValueError, TypeError):
        return "INVALID_DATE"


## 4. Retrieve and review candidate samples
The `samples` table is the title-matched (or explicitly included) list. Review it before continuing if you want to adjust the selection. To change it, edit configuration and rerun from configuration onward.


In [4]:
all_depositions = list(api_pages("deposition"))
lookup = {}
for record in all_depositions:
    igsn = clean_igsn(record.get("igsn"))
    if not igsn:
        continue
    titles = (record.get("metadata") or {}).get("titles") or []
    title = " | ".join(t.get("title", "") for t in titles if isinstance(t, dict))
    lookup[igsn] = {"igsn": igsn, "title": title, "deposition_id": record["_id"]}

missing_explicit = set(EXTRA_IGSNS) - set(lookup)
if missing_explicit:
    raise ValueError(f"Explicit IGSNs have no accessible deposition: {sorted(missing_explicit)}")
sample_rows = []
for igsn, record in lookup.items():
    if not excluded(igsn) and (title_matches(record["title"]) or igsn in EXTRA_IGSNS):
        sample_rows.append({**record, "selection_basis":
                            "explicit_igsn" if igsn in EXTRA_IGSNS else "title_match"})
samples = pd.DataFrame(sample_rows, columns=["igsn", "title", "deposition_id", "selection_basis"])
samples = samples.sort_values("igsn").reset_index(drop=True)
if samples.empty:
    raise ValueError("No candidate samples. Review title terms, permissions, or EXTRA_IGSNS.")
matched_ids = sorted(samples["igsn"], key=len, reverse=True)


def title_source(igsn):
    if excluded(igsn):
        return None
    return next((candidate for candidate in matched_ids
                 if is_descendant(igsn, candidate)), None)

print(f"{len(all_depositions):,} accessible depositions; {len(samples):,} candidates")
display(samples)


4,606 accessible depositions; 1,866 candidates


,igsn,title,deposition_id,selection_basis
0,APLMAL00003,Ti64 Etched Sample 1,6a6cb96fd64c1367b86d16b3,title_match
1,APLMAL00004,Ti64 Etched Sample 2,6a6cbae4c443dd8b646272dd,title_match
2,APLMAL00005,Ti64 Etched Sample 3,6a6cbc76c443dd8b646272e0,title_match
3,APLMAL00006,Titan23 Mech Polished,6a6cfc7fa7a7b6e805c0dc5f,title_match
4,APLMAL00006-001,Titan23 Mech Polished - 001,6a6cfc7fa7a7b6e805c0dc63,title_match
...,...,...,...,...
1861,NWXMAI00003,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807c45531e775ae73d39d4a,title_match
1862,NWXMAI00004,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807c4a7dd7c08f074d39d44,title_match
1863,NWXMAI00009,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807c8a331e775ae73d39d4e,title_match
1864,NWXMAI00010,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807cb8c0f5e297109d39d88,title_match


## 5. Scan raw-data metadata
Each data type is scanned once. Matching is done locally, including child IGSNs. Scan totals refer to accessible items, not underlying files. Items without IGSNs cannot be assigned to a candidate and are counted in the diagnostics.


In [5]:
available_types = api_get("aimdl/datatype")
missing_types = set(DATA_TYPES) - set(available_types)
if missing_types:
    raise ValueError(f"Requested data types not available: {sorted(missing_types)}")
raw_items = {}
scan_rows = []
for data_type, technique in DATA_TYPES.items():
    scanned = matched = missing_igsn = 0
    for item in api_pages("aimdl/datafiles", {
        "dataType": data_type,
        "extraFields": json.dumps(["meta.igsn", "meta.experiment_date"]),
    }):
        scanned += 1
        igsn = clean_igsn((item.get("meta") or {}).get("igsn"))
        if not igsn:
            missing_igsn += 1
        elif title_source(igsn) is not None:
            raw_items[item["_id"]] = (technique, item)
            matched += 1
        if scanned % 1000 == 0:
            print(f"{data_type}: {scanned:,} scanned, {matched:,} matched", flush=True)
    scan_rows.append({"data_type": data_type, "scanned_items": scanned,
                      "matched_items": matched, "items_without_igsn": missing_igsn})
    print(f"{data_type}: complete — {scanned:,} scanned, {matched:,} matched", flush=True)
scan_diagnostics = pd.DataFrame(scan_rows)
display(scan_diagnostics)


xrd_raw: 1,000 scanned, 678 matched
xrd_raw: 2,000 scanned, 696 matched
xrd_raw: 3,000 scanned, 778 matched
xrd_raw: 4,000 scanned, 878 matched
xrd_raw: 5,000 scanned, 878 matched
xrd_raw: 6,000 scanned, 878 matched
xrd_raw: 7,000 scanned, 878 matched
xrd_raw: 8,000 scanned, 945 matched
xrd_raw: 9,000 scanned, 1,789 matched
xrd_raw: 10,000 scanned, 2,683 matched
xrd_raw: 11,000 scanned, 3,007 matched
xrd_raw: 12,000 scanned, 3,460 matched
xrd_raw: 13,000 scanned, 4,153 matched
xrd_raw: 14,000 scanned, 4,833 matched
xrd_raw: 15,000 scanned, 5,141 matched
xrd_raw: 16,000 scanned, 5,337 matched
xrd_raw: 17,000 scanned, 6,251 matched
xrd_raw: 18,000 scanned, 7,251 matched
xrd_raw: 19,000 scanned, 8,082 matched
xrd_raw: 20,000 scanned, 8,675 matched
xrd_raw: 21,000 scanned, 9,594 matched
xrd_raw: 22,000 scanned, 10,501 matched
xrd_raw: 23,000 scanned, 11,497 matched
xrd_raw: 24,000 scanned, 12,497 matched
xrd_raw: 25,000 scanned, 13,493 matched
xrd_raw: 26,000 scanned, 14,493 matched
xrd_ra

,data_type,scanned_items,matched_items,items_without_igsn
0,xrd_raw,49130,31556,0
1,pdv_trace,10937,7524,0


## 6. Enumerate actual files
Counts include every stored file inside an item classified as raw XRD or PDV; auxiliary files within such an item also count. The detailed inventory provides file names for auditing. Distinct stored copies have distinct IDs and count separately.

If interrupted, rerun this cell in the same kernel to reuse already completed listings. Restart the kernel for a fresh file listing after server-side changes. No failures are converted into zeros.


In [6]:
# In-memory cache is scoped by API URL and user ID; credentials are not cached here.
cache_scope = (api_url, user["_id"])
if globals().get("_file_cache_scope") != cache_scope:
    _file_cache = {}
    _file_cache_scope = cache_scope

file_rows, item_rows = [], []
for number, (item_id, (technique, item)) in enumerate(raw_items.items(), 1):
    meta = item.get("meta") or {}
    igsn = clean_igsn(meta.get("igsn"))
    source = title_source(igsn)
    if source is None:
        raise RuntimeError(f"Selection changed; rerun the scan before enumeration: {igsn}")
    if item_id not in _file_cache:
        _file_cache[item_id] = list(api_pages(f"item/{item_id}/files"))
    item_files = _file_cache[item_id]
    common = {
        "igsn": igsn,
        "title": lookup.get(igsn, {}).get("title", ""),
        "title_source_igsn": source,
        "matched_title": lookup[source]["title"],
        "match_relation": "exact" if source == igsn else "child_of_title_match",
        "experiment_date": experiment_day(meta.get("experiment_date")),
        "experiment_date_original": meta.get("experiment_date"),
        "technique": technique,
        "item_id": item_id,
        "item_name": item.get("name", ""),
    }
    item_rows.append({**common, "stored_file_count": len(item_files)})
    for record in item_files:
        file_rows.append({**common, "file_id": record["_id"],
                          "file_name": record.get("name", "")})
    if number % 100 == 0 or number == len(raw_items):
        print(f"Listed {number:,}/{len(raw_items):,} items; {len(file_rows):,} files", flush=True)

common_columns = ["igsn", "title", "title_source_igsn", "matched_title", "match_relation",
                  "experiment_date", "experiment_date_original", "technique", "item_id", "item_name"]
items_inventory = pd.DataFrame(item_rows, columns=common_columns + ["stored_file_count"])
files = pd.DataFrame(file_rows, columns=common_columns + ["file_id", "file_name"])
files = files.drop_duplicates(["igsn", "technique", "file_id"])
print(f"Completed: {len(items_inventory):,} items and {len(files):,} distinct file associations")
display(files.head(10))


Listed 100/39,080 items; 100 files
Listed 200/39,080 items; 200 files
Listed 300/39,080 items; 300 files
Listed 400/39,080 items; 400 files
Listed 500/39,080 items; 500 files
Listed 600/39,080 items; 600 files
Listed 700/39,080 items; 700 files
Listed 800/39,080 items; 800 files
Listed 900/39,080 items; 900 files
Listed 1,000/39,080 items; 1,000 files
Listed 1,100/39,080 items; 1,100 files
Listed 1,200/39,080 items; 1,200 files
Listed 1,300/39,080 items; 1,300 files
Listed 1,400/39,080 items; 1,400 files
Listed 1,500/39,080 items; 1,500 files
Listed 1,600/39,080 items; 1,600 files
Listed 1,700/39,080 items; 1,700 files
Listed 1,800/39,080 items; 1,800 files
Listed 1,900/39,080 items; 1,900 files
Listed 2,000/39,080 items; 2,000 files
Listed 2,100/39,080 items; 2,100 files
Listed 2,200/39,080 items; 2,200 files
Listed 2,300/39,080 items; 2,300 files
Listed 2,400/39,080 items; 2,400 files
Listed 2,500/39,080 items; 2,500 files
Listed 2,600/39,080 items; 2,600 files
Listed 2,700/39,080 it

KeyboardInterrupt: 

## 7. Summary and completeness checks
One row per actual IGSN and recorded experiment date. `UNKNOWN_DATE` / `INVALID_DATE` rows retain files whose dates cannot be assigned. Zero counts mean no associated stored files of that type in the retrieved inventory, not proof that no experiment occurred.

`candidate_coverage` checks each candidate and all included descendants, even when a child's own title was selected. Candidates with no raw items are listed separately rather than assigned fabricated experiment dates.


In [ ]:
keys = ["igsn", "experiment_date"]
count_columns = ["xrd_file_count", "pdv_file_count"]
labels = ["igsn", "title", "title_source_igsn", "matched_title", "match_relation"]
if items_inventory.empty:
    summary = pd.DataFrame(columns=keys + labels[1:] + count_columns)
else:
    pairs = items_inventory[keys].drop_duplicates()
    if files.empty:
        summary = pairs.assign(xrd_file_count=0, pdv_file_count=0)
    else:
        counts = (files.groupby(keys + ["technique"])["file_id"].nunique()
                  .unstack("technique", fill_value=0)
                  .reindex(columns=["XRD", "PDV"], fill_value=0)
                  .rename(columns={"XRD": "xrd_file_count", "PDV": "pdv_file_count"})
                  .rename_axis(columns=None).reset_index())
        summary = pairs.merge(counts, on=keys, how="left", validate="one_to_one")
        summary[count_columns] = summary[count_columns].fillna(0).astype(int)
    summary = summary.merge(items_inventory[labels].drop_duplicates("igsn"),
                            on="igsn", how="left", validate="many_to_one")
    summary = summary[keys + labels[1:] + count_columns].sort_values(keys).reset_index(drop=True)

coverage_rows = []
for candidate in samples.to_dict("records"):
    mask = items_inventory["igsn"].map(lambda x: is_descendant(x, candidate["igsn"]))
    selected = items_inventory.loc[mask.astype(bool)]
    coverage_rows.append({**candidate, "matched_raw_items": len(selected),
                          "stored_files": int(selected["stored_file_count"].sum()),
                          "status": "no_raw_items" if selected.empty else
                                    ("items_without_files" if selected["stored_file_count"].sum() == 0
                                     else "files_found")})
candidate_coverage = pd.DataFrame(coverage_rows)
date_issues = items_inventory.loc[items_inventory["experiment_date"].isin(["UNKNOWN_DATE", "INVALID_DATE"])]
empty_items = items_inventory.loc[items_inventory["stored_file_count"] == 0]

# Ensure aggregation did not lose file associations, including unknown dates.
assert int(summary[count_columns].sum().sum()) == len(files)
print(f"Summary rows: {len(summary):,}; date issues: {len(date_issues):,}; empty items: {len(empty_items):,}")
display(summary)
print("Candidate coverage:")
display(candidate_coverage)
if not date_issues.empty:
    print("Items requiring date review:")
    display(date_issues.head(20))


## 8. Export CSVs and run settings
Creates a new local directory with summary, per-file inventory, per-item inventory, candidate coverage, and diagnostic CSVs. Credentials are never written to exports. These CSVs can be opened in Excel or loaded with pandas. Counts can change as new data are ingested; the API does not provide a transactionally frozen snapshot across pages.


In [ ]:
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = OUTPUT_BASE / run_stamp
output_dir.mkdir(parents=True, exist_ok=False)
exports = {
    "titanium_experiment_summary.csv": summary,
    "titanium_files.csv": files,
    "titanium_items.csv": items_inventory,
    "titanium_candidates.csv": samples,
    "candidate_coverage.csv": candidate_coverage,
    "scan_diagnostics.csv": scan_diagnostics,
    "date_issues.csv": date_issues,
    "items_without_files.csv": empty_items,
}
for filename, frame in exports.items():
    frame.to_csv(output_dir / filename, index=False)
settings = {
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "api_url": api_url,
    "include_children": INCLUDE_CHILDREN,
    "title_terms_case_sensitive": TITLE_TERMS_CASE_SENSITIVE,
    "title_terms_case_insensitive": TITLE_TERMS_CASE_INSENSITIVE,
    "extra_igsns": EXTRA_IGSNS, "exclude_igsns": EXCLUDE_IGSNS,
    "data_types": DATA_TYPES,
    "date_rule": "calendar date as recorded; no timezone conversion",
    "count_rule": "distinct stored file IDs within raw-classified items",
    "summary_rows": len(summary), "file_associations": len(files),
}
(output_dir / "run_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
print("Saved outputs to:", output_dir.resolve())


## Troubleshooting and interpretation
- **Import error:** run the optional install cell and restart the kernel if necessary.
- **Authentication failure / HTTP 401:** check `.env` or restart and enter your API key at the private prompt.
- **HTTP 403:** your account lacks access to a requested resource. The run stops; it does not treat denied data as absent.
- **No candidates:** inspect the terms and sample permissions; alloy designations may omit “Ti”.
- **No data for a candidate:** title records can exist outside the AIMD-L collection scanned by these endpoints, and data may be unclassified or missing IGSNs.
- **Unknown dates:** inspect `date_issues.csv`; upload/creation timestamps are intentionally not used as replacements.
- **Long enumeration:** every selected item requires a file-list request; let progress continue. Rerun the enumeration cell after interruption to reuse completed lists in this kernel.
- **Changed selection:** rerun from the candidate cell onward. Changing API URL or login requires rerunning authentication too.
- **Fresh data:** restart and Run All to discard the in-memory file-list cache.

References: [AIMD-L tutorial](https://github.com/openmsi/aimdl_examples/blob/main/tutorials/01_api_endpoints.ipynb), [HTMDEC API](https://data.htmdec.org/api/v1), [IGSN documentation](https://docs.htmdec.org/docs/users_guide/igsn.html).
